# GDS Enrichment Closes the Gaps

This notebook is the positive-case companion to the three gap demonstrations in this directory.

Those notebooks show that Genie **fails** when the tables contain only raw transaction data — hub detection returns no score, community detection returns only bilateral pairs, and merchant overlap returns high-volume normal accounts instead of ring members. Each failure is a structural property of SQL, not a limitation of Genie.

This notebook confirms the other side: after GDS enrichment adds `risk_score` (PageRank), `community_id` (Louvain), and `similarity_score` (Node Similarity) to the account feature tables, **Genie answers the same class of questions accurately**.

**Three checks:**

- **Check 1** — PageRank: hub detection with a score (precision at top-20 > 70%)
- **Check 2** — Louvain: community structure visible (max ring coverage > 80%)
- **Check 3** — Node Similarity: ring pairs surface (same-ring fraction > 60%)

---

### Genie Space Setup

Connect the **after-enrichment** Genie Space to these tables before running:

| Table | Type | Used by |
|-------|------|---------|
| `gold_accounts` | Gold — account metadata + `risk_score`, `community_id`, `similarity_score` | Check 1, Check 2 |
| `gold_account_similarity_pairs` | Gold — Jaccard similarity pairs from Node Similarity | Check 3 |
| `transactions` | Base — account-to-merchant payment events | Combined queries |
| `account_links` | Base — account-to-account transfer edges | Combined queries |
| `merchants` | Base — merchant dimension | Combined queries |
| `account_labels` | Base — fraud ground-truth labels | Combined queries |

The gold tables are created by `feature_engineering/03_pull_gold_tables` sections 6–7.
The base tables are created by `setup/upload_and_create_tables.sh`.

---

**Before running:** set your `SPACE_ID` in the configuration cell below.

In [ ]:
%pip install --upgrade databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
SPACE_ID = "YOUR-GENIE-SPACE-ID"  # <-- replace this

CATALOG  = "graph-enriched-lakehouse"
SCHEMA   = "graph-enriched-schema"
VOLUME   = "graph-enriched-volume"

# Demo gold tables written by 03_pull_gold_tables sections 6–7.
# Connect your after-enrichment Genie Space to both of these tables.
GOLD_FEATURES_TABLE = f"{CATALOG}.{SCHEMA}.gold_accounts"
GOLD_PAIRS_TABLE    = f"{CATALOG}.{SCHEMA}.gold_account_similarity_pairs"

In [ ]:
from databricks.sdk import WorkspaceClient
from demo_utils import (
    genie_caller,
    load_ground_truth,
    check_risk_score_precision,
    check_community_purity,
    check_ring_pair_fraction,
)

# ── Connect and bind Genie caller ─────────────────────────────────────────────
w = WorkspaceClient()
print("Connected to:", w.config.host)

ask_genie = genie_caller(w, SPACE_ID)

In [ ]:
# ── Pre-flight: verify demo gold tables exist ─────────────────────────────────
all_present = True
for tbl in [GOLD_FEATURES_TABLE, GOLD_PAIRS_TABLE]:
    try:
        w.tables.get(full_name=tbl)
        print(f"[OK]      {tbl}")
    except Exception:
        print(f"[MISSING] {tbl}  — run 03_pull_gold_tables sections 6-7 first")
        all_present = False

if not all_present:
    raise SystemExit("One or more demo gold tables are missing. Run 03_pull_gold_tables before continuing.")

In [ ]:
# ── Load ground truth ────────────────────────────────────────────────────────
_GT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/ground_truth.json"
gt = load_ground_truth(_GT_PATH)
ring_lists = [r["account_ids"] for r in gt["rings"]]

print(
    f"Ground truth: {gt['summary']['total_rings']} rings, "
    f"{gt['summary']['total_fraud_accounts']:,} fraud accounts, "
    f"{gt['summary']['total_whale_accounts']:,} whales"
)

## Check 1 — PageRank: Hub Detection With a Score

After GDS enrichment the `gold_accounts` table gains a continuous `risk_score` column derived from PageRank. Ring members score high because they receive transfers from other high-scoring ring members — the centrality compounds recursively through the ring topology. Whale accounts score moderate because their senders are peripheral low-PageRank nodes.

The question below asks for the highest-risk accounts. With `risk_score` available, Genie can sort by it and return a ranked list where the top accounts are genuinely high-risk rather than merely high-volume.

**Expected result:** the top 20 returned accounts should be predominantly fraud ring members, not whale accounts. Precision at top-20 should exceed 70%.

In [ ]:
response_pr = ask_genie(
    "Which accounts have the highest fraud risk based on their transfer network position?"
)

if response_pr["text"] and response_pr["df"] is None:
    print("Genie returned a text response:")
    print(response_pr["text"])
else:
    print(f"Status: {response_pr['status']}  |  Rows returned: {len(response_pr['df'])}")
    print()
    print("SQL Genie generated:")
    print(response_pr["sql"])
    print()
    print("Genie's result:")
    display(response_pr["df"])

## Validate — Precision at Top-20

`check_risk_score_precision` sorts Genie's result by the score column descending, takes the top 20 accounts, and labels each against ground truth. Precision is the fraction of those 20 accounts that are genuine fraud ring members.

Pass criterion: **precision > 70%**.

In [ ]:
if response_pr["df"] is None:
    print("Genie returned a text response — no tabular result to validate.")
    result_pr = {"passed": False, "precision": 0.0, "true_positives": 0, "topn": 20}
else:
    result_pr = check_risk_score_precision(response_pr["df"], gt, topn=20)

    print("Precision at top-20:")
    print(f"  Fraud accounts in top-20: {result_pr['true_positives']} of {result_pr['topn']}")
    print(f"  Precision: {result_pr['precision']:.1%}")
    print()
    if result_pr["passed"]:
        print("PASSED — risk_score separates fraud ring members from whale accounts")
    else:
        print("FAILED — precision below 70%; check that GDS enrichment has run")

## Why risk_score Enables Threshold Analysis

Before GDS enrichment, Genie returned a flat list of accounts with no score. An analyst could not tell which accounts were fraud ring members and which were high-volume legitimate hubs. There was no value to cut on, no tradeoff curve to plot, and no F1 score to optimize without external ground truth.

After GDS enrichment the `risk_score` column changes what is possible:

- **A threshold exists.** An analyst can set a cutoff of 0.7 or 0.8 and observe which accounts appear above it.
- **The tradeoff is visible.** Moving the threshold up reduces false positives and increases false negatives. Moving it down does the reverse. The curve is measurable.
- **Separation is visible without ground truth.** The score distribution shows a gap between high-PageRank ring members and moderate-PageRank whale accounts. An analyst can see the bimodal shape without needing to know which accounts are fraud.

The precision number above confirms that the top of the ranked list is dominated by ring members. In `hub_detection_no_threshold.ipynb`, the same question returned a mixed list at roughly 40–50% precision with no way to improve it. The `risk_score` column is the only difference.

## Check 2 — Louvain: Community Structure Visible

After GDS enrichment the `gold_accounts` table gains a `community_id` column derived from Louvain community detection. Each fraud ring is assigned a stable community ID that groups all 100 ring members together. Accounts in different rings receive different IDs.

The question below asks Genie to find groups of accounts that form suspicious transaction communities. With `community_id` available, Genie can return accounts grouped by that column rather than returning only bilateral transfer pairs.

**Expected result:** Genie returns a grouped result where each identified community covers 80% or more of a single fraud ring. The `max_ring_coverage` metric should exceed 80%.

In [ ]:
response_lv = ask_genie(
    "Find groups of accounts that form suspicious transaction communities "
    "based on their transfer patterns."
)

if response_lv["text"] and response_lv["df"] is None:
    print("Genie returned a text response:")
    print(response_lv["text"])
else:
    print(f"Status: {response_lv['status']}  |  Rows returned: {len(response_lv['df'])}")
    print()
    print("SQL Genie generated:")
    print(response_lv["sql"])
    print()
    print("Genie's result:")
    display(response_lv["df"])

## Validate — Community Ring Coverage

`check_community_purity` detects the grouping column in Genie's result, then measures what fraction of each fraud ring is covered by a single identified group. The `max_ring_coverage` value is the best coverage achieved across all rings.

Pass criterion: **max_ring_coverage > 80%**.

In [ ]:
if response_lv["df"] is None:
    print("Genie returned a text response — no tabular result to validate.")
    result_lv = {"passed": False, "max_ring_coverage": 0.0, "groups_returned": 0, "total_rows": 0, "structure_type": "none"}
else:
    result_lv = check_community_purity(response_lv["df"], ring_lists)

    print("Community purity analysis:")
    print(f"  Structure type:    {result_lv['structure_type']}")
    print(f"  Groups returned:   {result_lv['groups_returned']}")
    print(f"  Total rows:        {result_lv['total_rows']}")
    print()
    print(f"  Max ring coverage: {result_lv['max_ring_coverage']:.1%}")
    print(f"  Pass threshold:    > 80%")
    print()
    if result_lv["passed"]:
        print("PASSED — community_id exposes full fraud ring structure")
    else:
        print("FAILED — coverage below 80%; check that Louvain has written community_id")

## Why community_id Exposes Full Ring Structure

Before GDS enrichment, Genie could only return bilateral pairs — accounts that transfer money back and forth. SQL transitive closure merges any rings that share a cross-ring edge, collapsing multiple rings into one giant component. Louvain's modularity optimization keeps them separated.

After GDS enrichment the `community_id` column encodes the result of that global optimization directly in the table:

- **Each ring becomes a single community.** All 100 members share the same `community_id`, so a `GROUP BY community_id` query returns full rings rather than bilateral pairs.
- **Ring boundaries are preserved.** Louvain maximizes within-ring edge density while minimizing cross-ring edges, so two rings that happen to share a transfer between them are not merged into one community.
- **The result is actionable.** An analyst reviewing a community of 100 accounts with the same `community_id` has a concrete fraud ring to investigate, not a list of account pairs to manually reconstruct into groups.

In `community_structure_invisible.ipynb`, the same question returned only bilateral pairs with `max_ring_coverage = 0.0`. The `community_id` column is the only difference.

## Check 3 — Node Similarity: Ring Pairs Surface

After GDS enrichment, Node Similarity computes the Jaccard similarity between every pair of accounts based on their shared merchant visit sets and writes the results as `SIMILAR_TO` relationships in Neo4j. The `gold_account_similarity_pairs` gold table materialises those relationships as `(account_id_a, account_id_b, similarity_score)` rows. Ring members share 5 anchor merchants out of roughly 30 total merchants each, giving a Jaccard score around 9%. High-volume normal accounts may share 60 merchants but visit 200–400 each, giving a Jaccard score around 8% — similar to ring pairs in absolute terms but lower in the normalized score.

The question below asks Genie for account pairs with the most similar merchant patterns. With `gold_account_similarity_pairs` available, Genie can sort by the Jaccard-normalized `similarity_score` rather than raw shared count.

**Expected result:** the top 20 returned pairs should be predominantly same-ring account pairs. The same-ring fraction should exceed 60%.

In [ ]:
response_ns = ask_genie(
    "Which pairs of accounts share the most similar merchant visit patterns?"
)

if response_ns["text"] and response_ns["df"] is None:
    print("Genie returned a text response:")
    print(response_ns["text"])
else:
    print(f"Status: {response_ns['status']}  |  Rows returned: {len(response_ns['df'])}")
    print()
    print("SQL Genie generated:")
    print(response_ns["sql"])
    print()
    print("Genie's result:")
    display(response_ns["df"])

## Validate — Same-Ring Pair Fraction

`check_ring_pair_fraction` classifies each returned pair as same-ring (both accounts in the same fraud ring), cross-ring, or unknown (at least one account not in any ring). The `same_ring_fraction` is the fraction of total pairs that are same-ring.

Pass criterion: **same_ring_fraction > 60%**.

In [ ]:
if response_ns["df"] is None:
    print("Genie returned a text response — no tabular result to validate.")
    result_ns = {"passed": False, "same_ring_fraction": 0.0, "total_pairs": 0,
                 "same_ring_pairs": 0, "cross_ring_pairs": 0, "unknown_pairs": 0, "rings_touched": 0}
else:
    df_ns = response_ns["df"]

    # ── Identify account ID columns ───────────────────────────────────────────────
    col_a = next(
        (c for c in df_ns.columns if "account_id_a" in c.lower() or (c.lower().endswith("_a") and "account" in c.lower())),
        None,
    )
    col_b = next(
        (c for c in df_ns.columns if "account_id_b" in c.lower() or (c.lower().endswith("_b") and "account" in c.lower())),
        None,
    )
    if col_a is None or col_b is None:
        cols = df_ns.columns.tolist()
        col_a, col_b = cols[0], cols[1]

    pairs = list(zip(df_ns[col_a].astype(int), df_ns[col_b].astype(int)))

    result_ns = check_ring_pair_fraction(pairs, ring_lists)

    print("Same-ring pair fraction analysis:")
    print(f"  Total pairs returned:   {result_ns['total_pairs']}")
    print(f"  Same-ring pairs:        {result_ns['same_ring_pairs']}")
    print(f"  Cross-ring pairs:       {result_ns['cross_ring_pairs']}")
    print(f"  Unknown pairs:          {result_ns['unknown_pairs']}")
    print(f"  Rings touched:          {result_ns['rings_touched']}")
    print()
    print(f"  Same-ring fraction:     {result_ns['same_ring_fraction']:.1%}")
    print(f"  Pass threshold:         > 60%")
    print()
    if result_ns["passed"]:
        print("PASSED — similarity_score surfaces same-ring pairs over volume-inflated normals")
    else:
        print("FAILED — fraction below 60%; check that Node Similarity has written similarity_score")

## Why gold_account_similarity_pairs Overcomes Volume Inflation

Before GDS enrichment, Genie ranked account pairs by raw shared-merchant count. High-volume normal accounts visit 200–400 merchants and accumulate 40–80 shared merchants with other high-volume accounts by chance, dominating the top of the list. Ring members share only 4–5 anchor merchants and appear far down the ranking.

After GDS enrichment the `gold_account_similarity_pairs` table stores Jaccard-normalized scores for every pair:

- **Volume inflation disappears.** A pair sharing 60 merchants out of a combined 740 scores 8% Jaccard. A ring pair sharing 5 merchants out of a combined 55 scores 9% Jaccard. The ring pair wins even though it has far fewer shared merchants in absolute terms.
- **Anchor merchants become signal.** The 5 shared anchor merchants that define each fraud ring are a concentrated signal in a small merchant footprint. Jaccard amplifies concentration rather than suppressing it.
- **The result is complete.** Node Similarity computes pairwise Jaccard for every account in the graph simultaneously — not just the top-volume accounts that a raw-count query would surface. Ring pairs that are not the highest-volume accounts in the dataset still appear near the top of the similarity ranking.

In `merchant_overlap_volume_inflation.ipynb`, the same question returned high-volume normal account pairs with `same_ring_fraction` below 30%. The `gold_account_similarity_pairs` table is the only difference.

In [ ]:
print("=" * 62)
print("GDS ENRICHMENT VALIDATION SUMMARY")
print("=" * 62)
print()

checks = [
    ("PageRank — Hub Detection",      result_pr["passed"], f"precision at top-{result_pr['topn']}: {result_pr['precision']:.1%}"),
    ("Louvain — Community Structure", result_lv["passed"], f"max ring coverage: {result_lv['max_ring_coverage']:.1%}"),
    ("Node Similarity — Ring Pairs",  result_ns["passed"], f"same-ring fraction: {result_ns['same_ring_fraction']:.1%}"),
]

for name, passed, detail in checks:
    status = "PASSED" if passed else "FAILED"
    print(f"  {status}  {name}")
    print(f"          {detail}")
    print()

all_passed = all(p for _, p, _ in checks)
print("-" * 62)
if all_passed:
    print("All three checks passed — GDS enrichment closes all three gaps.")
else:
    failed = [n for n, p, _ in checks if not p]
    print(f"Check(s) failed: {', '.join(failed)}")
    print("Verify that GDS algorithms have run and written their output columns.")
print("-" * 62)